# H-002 · Discovery multiple-testing (raw vs BH-FDR)

Raw `pvalue < 0.05` versus Benjamini–Hochberg FDR at 5% on the Pass A candidate CSV
restricted to `POOL_STAR`. **Do not re-run EG.** Soft N from stack / 35.

Do not open the sealed holdout in this notebook.

Next: run H-003.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    arm_selection_table,
    fold_table,
    fold_val_metrics,
    full_is_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    apply_bh_fdr,
    candidates_path,
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_s2_cuts,
    load_universe_panels,
    lookbacks_for_bar,
    n_trials_local,
    n_trials_stack,
    overlay_ols_hedge,
    pairs_path,
    register_hypothesis_arms,
    s2_artifacts_dir,
    s2_data_dir,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds
from strategies.s2_coint.book import select_book

VERSION = "v2"
DATA_DIR = s2_data_dir(ROOT, version=VERSION)
STAR_PATH = os.path.join(s2_artifacts_dir(version=VERSION, root=ROOT), "s2_star_stack.json")
os.makedirs(os.path.dirname(STAR_PATH), exist_ok=True)
cuts = load_s2_cuts(root=ROOT, version=VERSION)
DROP_RATE = cuts.get("yahoo_drop_rate")
print("STAR_PATH", STAR_PATH)
print("yahoo_drop_rate", DROP_RATE)
print("RESEARCH_IS_END", cuts.get("RESEARCH_IS_END"))

stack = load_star_stack(STAR_PATH)
require_star("POOL_STAR", stack.get("POOL_STAR"))
POOL_STAR = str(stack["POOL_STAR"])
SOFT_N = int(stack.get("N_STAR") or 35)
BAR = str(stack.get("BAR_STAR") or "1d")
FDR_ARMS = ("raw_p05", "bh_fdr")
HYP_ID = "H-002"
print("POOL_STAR", POOL_STAR, "soft N", SOFT_N)


## 1. Filter candidates by pooling + FDR arm


In [ ]:
cand = pd.read_csv(candidates_path(BAR, version=VERSION, root=ROOT))
if POOL_STAR == "industry":
    base = cand.loc[cand["same_industry"].astype(bool)].copy()
else:
    base = cand.loc[cand["same_sector"].astype(bool)].copy()
base["pvalue"] = pd.to_numeric(base["pvalue"], errors="coerce")

def book_from(df: pd.DataFrame, n: int) -> list[str]:
    leaf = "industry_y" if POOL_STAR == "industry" else "sector_y"
    pool_map = {str(r.pair_id): str(getattr(r, leaf)) for r in df.itertuples(index=False)}
    chosen = select_book(
        df, pool_of_pair=pool_map, per_pool_cap=max(n, 10), global_cap=n, pvalue_threshold=1.0
    )
    # select_book still filters pvalue < threshold; pass 1.0 then filter outside for FDR arm
    return [str(p) for p in chosen["pair_id"]]

raw = base.loc[base["eligible"].astype(bool) & (base["pvalue"] < 0.05)].copy()
fdr = apply_bh_fdr(base.loc[base["eligible"].astype(bool)].copy(), alpha=0.05)
fdr = fdr.loc[fdr["fdr_pass"]].copy()
books = {
    "raw_p05": book_from(raw, SOFT_N),
    "bh_fdr": book_from(fdr, SOFT_N),
}
for k, v in books.items():
    print(k, len(v), v[:5])


## 2. Walk-forward score


In [ ]:
union_ids = sorted(set(books["raw_p05"]) | set(books["bh_fdr"]))
train, full = load_universe_panels(None, BAR, union_ids, root=ROOT, version=VERSION)
lb0 = lookbacks_for_bar(BAR)
train = overlay_ols_hedge(train, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                          hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
is_end = pd.Timestamp(stack.get("RESEARCH_IS_END_STAR") or cuts["RESEARCH_IS_END"])
is_panel = train.copy()
s1_weekly = load_s1_weekly(ROOT)
dates = pd.DatetimeIndex(pd.to_datetime(is_panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
display(fold_table(folds))
ledger = os.path.join(os.path.dirname(STAR_PATH), "s2_variant_ledger.json")
register_hypothesis_arms(HYP_ID, list(FDR_ARMS), overwrite=True, path=ledger)
fold_parts, full_parts = [], []
for arm in FDR_ARMS:
    p = is_panel.loc[is_panel["pair_id"].astype(str).isin(set(books[arm]))].copy()
    cfg = {arm: config_from_stack(stack, bar=BAR, break_mode=stack.get("BREAK_STAR") or "off")}
    fold_parts.append(fold_val_metrics(p, folds, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
    full_parts.append(full_is_metrics(p, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
fold_df = pd.concat(fold_parts, ignore_index=True)
full_is_df = pd.concat(full_parts, ignore_index=True)
plot_fold_boxplots(fold_df, title="H-002 FDR fold-val")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
display(arm_selection_table(fold_df, full_is_df))


## 3. Type `FDR_STAR`


In [ ]:
FDR_STAR = ""  # type after the table: "raw_p05" | "bh_fdr"
require_star("FDR_STAR", FDR_STAR)
stack["FDR_STAR"] = FDR_STAR
stack["PAIRS_STAR"] = books[FDR_STAR]
pd.DataFrame({"pair_id": books[FDR_STAR], "rank": range(1, len(books[FDR_STAR]) + 1)}).to_csv(
    pairs_path(BAR, version=VERSION, root=ROOT), index=False
)
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH, "FDR_STAR", FDR_STAR)


## 4. Notes for next hyp

Next: run H-003.
